### Mugrade boilerplate

In [ ]:
### Run this cell to download and installs the necessary modules for the homework
!pip install --upgrade git+https://github.com/locuslab/mugrade.git
!wget -nc https://raw.githubusercontent.com/zkolter/llm_speedrun/refs/heads/main/part1_tokenizer_tests.py

import mugrade
import os
from part1_tokenizer_tests import *
os.environ["MUGRADE_HW"] = "Part 1 - Tokenizer"
os.environ["MUGRADE_KEY"] = "" ### Your key here

### BPE Training, Encoding, Decoding

For this portion of the homework, implement the BPE class we covered below.  You don't need to run the later pieces of code (they actually train the tokenize on the FineWeb-EDU dataset), but you are free to do so if you want to train your own tokenizer.  For your submission, you should filter word frequencies in training to ignore words that only occur once (we did both versions in class, but the test cases require the `train` function to filter words in that manner.)

Note that the mugrade local tests are commented by default, but you should uncomment them as you implement each function.  Unlike the previous assignment, we know that the details of each function likely won't make much sense without watching the lecture, but after watching the lecture the form of the function should be clear.

In [ ]:
import json
from collections import Counter
import re
from tqdm.auto import tqdm

class BPE:
    # @mugrade.local_tests
    def __init__(self, filename=None):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    @staticmethod
    # @mugrade.local_tests
    def merge_pair(word, a, b, merged):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def replace_special_tokens(self, word):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    
    # @mugrade.local_tests
    def train(self, text, target_vocab_size):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE
             
    # @mugrade.local_tests
    def encode(self, text):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def decode(self, tokens):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE
    
    # @mugrade.local_tests
    def save(self, fname):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

### Download datasets

The following code with download the (10B token subset of the) FineWeb-EDU dataset, plus the SmolTalk dataset.  To make this suitably fast, you shuld really get put a huggingface token in your .env file or equivalent for the notebook.

In [ ]:
from huggingface_hub import snapshot_download
import pyarrow.dataset as ds
import random
from dotenv import load_dotenv; load_dotenv()
import os
os.environ["HF_XET_HIGH_PERFORMANCE"]="1"

def download_dataset(name, subdir, field, output_file, chat=False):
    root = snapshot_download(name, repo_type="dataset", allow_patterns=subdir+"/*.parquet", max_workers=100)
    docs = ds.dataset(root + "/" + subdir, format="parquet").to_table()[field].to_pylist()
    random.seed(42)
    random.shuffle(docs)

    with open(output_file, "wt") as f:
        for doc in tqdm(docs):
            if chat:
                msgs = "".join([f"<{m["role"].upper()}>{m["content"]}</{m["role"].upper()}>" for m in doc])
                f.write(f"<DOCUMENT>{msgs}</DOCUMENT>")
            else:
                f.write(f"<DOCUMENT>{doc}</DOCUMENT>")

In [ ]:
download_dataset("HuggingFaceFW/fineweb-edu", "sample/10BT", "text", "fineweb-edu-10BT.shuffle2.txt")
download_dataset("HuggingFaceTB/smoltalk", "data/all", "messages", "smoltalk.shuffle.txt", chat=True)

### Train the tokenizer

Once you've downloaded the datasets, these next cells with train the tokenizer.  Note that this takes a long time, e.g. on my machine it takes about an hour.

In [ ]:
with open("fineweb-edu-10BT.shuffle2.txt", "rb") as f:
    text_train = f.read(40_000_000).decode("latin-1")
with open("smoltalk.shuffle.txt", "rb") as f:
    text_train += f.read(10_000_000).decode("latin-1")

bpe = BPE()
vocab_size = 2**15-256
bpe.special_tokens[vocab_size] = "<DOCUMENT>"
bpe.special_tokens[vocab_size+1] = "</DOCUMENT>"
bpe.special_tokens[vocab_size+2] = "<SYSTEM>"
bpe.special_tokens[vocab_size+3] = "</SYSTEM>"
bpe.special_tokens[vocab_size+4] = "<USER>"
bpe.special_tokens[vocab_size+5] = "</USER>"
bpe.special_tokens[vocab_size+6] = "<ASSISTANT>"
bpe.special_tokens[vocab_size+7] = "</ASSISTANT>"

bpe.train(text_train, vocab_size)
bpe.save("tokenizer_50M.bpe")

### Pretokenize the datasets

Finally, you can use the tokenizer to tokenize the entirety of our text datasets.  Since we'll be using them a lot during training, it's a good idea to take this approach and pre-convert the datasets to tokens that we can use directly for training, rather than doing the tokenization in real time.  This also takes a relatively long time, but since it is trivially parallelizable, if you have a machine with a lot of CPUs it can be done quickly for the entire datasets.

In [ ]:
from joblib import Parallel, delayed
from array import array
import os

def pretokenize(in_fname, out_fname, bpe, chunk_size = 20_000_000, tmpdir=".tmp"):
    def tokenize_chunk(i):
        with open(in_fname, "rb") as f:
            f.seek(i*chunk_size)
            tokens = bpe.encode(f.read(chunk_size).decode("latin-1"))
        with open(f"{tmpdir}/tokens.{i:05d}.bin", "wb") as f:
            array("H", tokens).tofile(f)

    n_chunks = (os.path.getsize(in_fname) + chunk_size - 1) // chunk_size
    jobs = Parallel(n_jobs=128, return_as="generator_unordered")(
        delayed(tokenize_chunk)(i) for i in range(n_chunks)
    )
    for _ in tqdm(jobs, total=n_chunks, smoothing=0):
        pass
    
    with open(out_fname, "wb") as fout:
        for i in tqdm(range(n_chunks)):
            with open(f"{tmpdir}/tokens.{i:05d}.bin", "rb") as fin:
                fout.write(fin.read())
    


In [ ]:
bpe = BPE("tokenizer_50M.bpe")
pretokenize("smoltalk.shuffle.txt", "smoltalk.shuffle.bin", bpe)
pretokenize("fineweb-edu-10BT.shuffle2.txt", "fineweb-edu-10BT.shuffle.bin", bpe)